In [1]:
!pip install mlflow boto3 optuna imbalanced-learn lightgbm

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import mlflow

In [3]:
import os

os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = 'NTsundere'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'd2d97678bd6872fa72b238e9d94b44144e0715d9'

In [4]:
mlflow.set_experiment("LightGBM HP Tuning")

2026/07/25 22:25:08 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/3be368ad0490414f977f555622a2549c', creation_time=1785007508638, experiment_id='6', last_update_time=1785007508638, lifecycle_stage='active', name='LightGBM HP Tuning', tags={}>

In [6]:
import pandas as pd

df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

C:\Users\iblad\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['category'])

In [9]:
# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

C:\Users\iblad\AppData\Roaming\Python\Python39\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [ ]:
# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

In [11]:
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):
    with mlflow.start_run():
        # Log model type and trial number
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

        return accuracy



In [12]:
def objective_lightgbm(trial):
    # Hyperparameter space to explore
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)  # L1 regularization
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True)  # L2 regularization

    # Log trial parameters
    params = {
        'n_estimators': n_estimators,
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'min_child_samples': min_child_samples,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda
    }

    # Create LightGBM model
    model = LGBMClassifier(n_estimators=n_estimators,
                           learning_rate=learning_rate,
                           max_depth=max_depth,
                           num_leaves=num_leaves,
                           min_child_samples=min_child_samples,
                           colsample_bytree=colsample_bytree,
                           subsample=subsample,
                           reg_alpha=reg_alpha,
                           reg_lambda=reg_lambda,
                           random_state=42)

    # Log each trial as a separate run in MLflow
    accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)

    return accuracy


In [13]:
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=100)  # Increased to 100 trials

    # Get the best parameters
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'],
                                learning_rate=best_params['learning_rate'],
                                max_depth=best_params['max_depth'],
                                num_leaves=best_params['num_leaves'],
                                min_child_samples=best_params['min_child_samples'],
                                colsample_bytree=best_params['colsample_bytree'],
                                subsample=best_params['subsample'],
                                reg_alpha=best_params['reg_alpha'],
                                reg_lambda=best_params['reg_lambda'],
                                random_state=42)

    # Log the best model with MLflow and print the classification report
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test, best_params, "Best")

    # Plot parameter importance
    optuna.visualization.plot_param_importances(study).show()

    # Plot optimization history
    optuna.visualization.plot_optimization_history(study).show()

In [ ]:
run_optuna_experiment()

[I 2026-07-25 22:26:19,316] A new study created in memory with name: no-name-cd59c705-fcc3-4189-beeb-7485f77b8d9f


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.033990 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98978
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 966
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


C:\Users\iblad\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/25 22:26:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 22:27:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
[I 2026-07-25 22:27:23,822] Trial 0 finished with value: 0.6854787571337984 and parameters: {'n_estimators': 151, 'learning_rate': 0.0008670745205158104, 'max_depth': 9, 'num_leaves': 36, 'min_child_samples': 40, 'colsample_bytree': 0.5078013968653607, 'subsample': 0.7523401039388797, 'reg_alpha': 0.0025680732096360495, 'reg_lambda': 0.00010327923008605154}. Best is trial 0 with value: 0.6854787571337984.


🏃 View run Trial_0_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/6/runs/5ed3d2cd6859407e8a652b3d4a2cb9ef
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/6
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031246 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 99046
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 974
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

C:\Users\iblad\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/25 22:28:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 22:28:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

C:\Users\iblad\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/25 22:52:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 22:52:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
[I 2026-07-25 22:52:36,887] Trial 16 finished with value: 0.7694990488268865 and parameters: {'n_estimators': 731, 'learning_rate': 0.010617159156951378, 'max_depth': 7, 'num_leaves': 66, 'min_child_samples': 30, 'colsample_bytree': 0.7012661473575376, 'subsample': 0.8720721050870696, 'reg_alpha': 0.004273057781056833, 'reg_lambda': 1.1111365278846186}. Best is trial 15 with value: 0.8193827943352356.


🏃 View run Trial_16_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/6/runs/aa31cbf8162f4626bb0f2d4688dfb4c3
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/6
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038528 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98870
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 960
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

C:\Users\iblad\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/25 22:53:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 22:53:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Trial_17_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/6/runs/ece3e40bd0bf4971acdd48708356f53c
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/6


[I 2026-07-25 22:53:59,692] Trial 17 finished with value: 0.8162122172902135 and parameters: {'n_estimators': 724, 'learning_rate': 0.09680543104379677, 'max_depth': 10, 'num_leaves': 72, 'min_child_samples': 58, 'colsample_bytree': 0.7132699650918488, 'subsample': 0.8217473600097437, 'reg_alpha': 0.007964962293467154, 'reg_lambda': 0.38205564589787216}. Best is trial 15 with value: 0.8193827943352356.
